In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import plotly.graph_objects as go
from src.features import FeatureEngineer
from src.regime import RegimeDetector
from src.strategies.trend_engine import TrendEngine
from src.strategies.mean_reversion import MeanReversionEngine
from src.risk.manager import RiskManager, RiskConfig
from src.portfolio import PortfolioController
from src.backtest import BacktestEngine

# 1. Load Data
df = pd.read_parquet('../data/raw/SPY.parquet')
df = df[~df.index.duplicated(keep='first')] # Dedup

# 2. Prepare Features
print("Engineering Features...")
fe = FeatureEngineer(df)
fe.add_volatility_features().add_trend_features().add_volume_features()
data = fe.get_features()

# Align prices with features
data['Close'] = fe.df.loc[data.index, 'Close']

# 3. Load Regime Model (Train on the spot for this test)
print("Training Regime Model...")
regime_engine = RegimeDetector(n_components=4)
# IMPORTANT: Map columns correctly
regime_engine.feature_cols = ['Vol_ratio', 'Momentum', 'Vol_short']
regime_engine.fit(data)

# 4. Initialize System
print("Initializing Engines...")
risk_manager = RiskManager(RiskConfig(target_volatility=0.20), 100000)
engines = [TrendEngine(), MeanReversionEngine()]
controller = PortfolioController(risk_manager, engines)
backtester = BacktestEngine(controller, regime_engine)

Engineering Features...
Training Regime Model...
Model trained. Converged: True
Initializing Engines...


In [2]:
# Run the simulation
# We pass data as a dictionary
results = backtester.run({'SPY': data})

# Calculate Buy & Hold for comparison
initial_price = data['Close'].iloc[0]
results['Buy_Hold'] = (data['Close'] / initial_price) * 100000

Starting Backtest on 2204 bars...
Backtest Complete. Final Equity: $180,965.27


In [3]:
# Plot Equity Curve vs Buy & Hold
fig = go.Figure()

# System Equity
fig.add_trace(go.Scatter(x=results.index, y=results['Equity'], 
                         mode='lines', name='Adaptive System', line=dict(color='lime')))

# Buy & Hold
fig.add_trace(go.Scatter(x=results.index, y=results['Buy_Hold'], 
                         mode='lines', name='SPY Buy & Hold', line=dict(color='gray', dash='dot')))

# Regime Background Colors (Optional but fancy)
# We can visualize where the system went to Cash
fig.update_layout(title='Adaptive System vs SPY (2015-2024)', template='plotly_dark')
fig.show()

# Plot Allocations
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=results.index, y=results['Allocation_Trend'], name='Trend Alloc', stackgroup='one'))
fig2.add_trace(go.Scatter(x=results.index, y=results['Allocation_MeanRev'], name='MeanRev Alloc', stackgroup='one'))
fig2.add_trace(go.Scatter(x=results.index, y=1-(results['Allocation_Trend']+results['Allocation_MeanRev']), name='Cash', stackgroup='one'))
fig2.update_layout(title='Dynamic Capital Allocation', template='plotly_dark')
fig2.show()